# Step 3: Data Analysis

Kilian Lüders & Hannah Birkenkötter

**Steps:**
1. Load Data
2. Validation: Chapter VII & Article 39
3. Analyse Variables
    1. Article 42 (No Matches)
    2. Article 25
    3. Chapter VII
    4. Article 41
    5. Article 39
    6. Article 39 language
    7. "Breach of International Peace"
    8. Decides / Demand / Authorizes
    9. Subset: Demand Request Decide
    10. Demand in resolutions without Decides / Chapter 7

**Input:**
- `data/CR-UNSC_2024-05-19_ALL_CSV_FULL.csv` – csv file from [https://zenodo.org/records/11212056] (we use this data at this point ot validate our own results).
- `data/full_data_vars.pkl` - full data with new variables on paragraph level (55065 rows x 21 columns).
- `data/data_res.pkl` - data with new variables on resolution level (2721 rows x 16 columns).

**Output:**
- `data/subset_demand_request_decide.pkl`


In [1]:
import pandas as pd

## 1. Load Data

In [2]:
# one observation per paragraph (clause)
full_data = pd.read_pickle("data/full_data_vars.pkl")
print(full_data.shape)

(55065, 21)


In [3]:
# one observation per resolution
dec_data = pd.read_pickle("data/data_res.pkl")
print(dec_data.shape)

(2721, 16)


In [4]:
# original masterdata from [https://zenodo.org/records/11212056]
masterdata = pd.read_csv("data/CR-UNSC_2024-05-19_ALL_CSV_FULL.csv")
masterdata['doc'] = masterdata['doc_id'].apply(lambda x: x.replace(".txt", "").replace("_EN", "").replace("_GOLD", ""))
masterdata['date'] = pd.to_datetime(masterdata.date)
print(masterdata.shape)

(2722, 83)


## 2. Validation: Chapter 7 & Article 39
Compare Masterdata from Fobbe et al with our variables

In [5]:
masterdata_var = masterdata[['doc', 'chapter7', 'peace_breach', 'peace_threat']].copy()
chpt7_df = dec_data.merge(masterdata_var, on="doc")
chpt7_df['chapter7'] = chpt7_df.chapter7.astype(int)
chpt7_df['peace_breach'] = chpt7_df.peace_breach.astype(int)
chpt7_df['peace_threat'] = chpt7_df.peace_threat.astype(int)
#chpt7_df['self_defence'] = chpt7_df.self_defence.astype(int)
chpt7_df.tail()

,doc,decides,decides_act,art25_charter,chpVII,art39_charter,art40_charter,art41_charter,art39lang,demand,authorizes,requests,int_peace_sec,art39lang_wout,year,decade,chapter7,peace_breach,peace_threat
2716,S_RES_2717_2023,1,1,0,1,0,0,0,1,1,1,1,1,1,2023,2020,1,0,0
2717,S_RES_2718_2023,1,0,0,0,0,0,0,0,0,0,1,0,0,2023,2020,0,0,0
2718,S_RES_2719_2023,1,0,0,0,0,0,0,1,0,1,1,1,0,2023,2020,0,0,0
2719,S_RES_2720_2023,0,1,0,0,0,0,0,0,1,0,1,0,0,2023,2020,0,0,0
2720,S_RES_2721_2023,0,1,0,0,0,0,0,0,0,0,1,0,0,2023,2020,0,0,0


In [6]:
chpt7_df[chpt7_df.chapter7 != chpt7_df.chpVII]

,doc,decides,decides_act,art25_charter,chpVII,art39_charter,art40_charter,art41_charter,art39lang,demand,authorizes,requests,int_peace_sec,art39lang_wout,year,decade,chapter7,peace_breach,peace_threat
216,S_RES_0217_1965,1,0,0,0,0,0,0,1,0,0,0,1,1,1965,1960,1,0,1
324,S_RES_0325_1973,1,0,0,0,0,0,0,1,0,0,1,1,0,1973,1970,1,0,0
446,S_RES_0447_1979,0,0,0,1,0,0,0,1,1,0,1,1,1,1979,1970,0,0,1
712,S_RES_0713_1991,1,1,0,1,0,0,0,1,0,0,0,1,1,1991,1990,0,0,1


In [7]:
cross_tab = pd.crosstab(
    chpt7_df['art39lang'],
    [chpt7_df['peace_breach'], chpt7_df['peace_threat']],
    dropna=False
)

# print cross table
print(cross_tab)

peace_breach     0       1   
peace_threat     0    1  0  1
art39lang                    
0             1621    0  0  0
1              675  420  4  1


In [8]:
chpt7_df[(chpt7_df.art39lang == 0) & (chpt7_df.peace_threat == 1)]

,doc,decides,decides_act,art25_charter,chpVII,art39_charter,art40_charter,art41_charter,art39lang,demand,authorizes,requests,int_peace_sec,art39lang_wout,year,decade,chapter7,peace_breach,peace_threat


## 3. Analyse Variables

In [9]:
# number of resolutions
dec_data[["decides", "decides_act", "art25_charter", "chpVII", "art39_charter", "art40_charter", "art41_charter", "art39lang", "demand", "authorizes", "requests"]].sum()

decides          1820
decides_act      1743
art25_charter      30
chpVII            896
art39_charter       6
art40_charter       6
art41_charter      49
art39lang        1100
demand            572
authorizes        295
requests         1909
dtype: int64

In [10]:
# number of clauses
full_data[["decides", "decides_act", "art25_charter", "chpVII", "art39_charter", "art40_charter", "art41_charter", "art39lang", "demand", "authorizes", "requests"]].sum()

decides          3986
decides_act      1743
art25_charter      36
chpVII            921
art39_charter       7
art40_charter       6
art41_charter      63
art39lang        1716
demand           1189
authorizes        523
requests         5985
dtype: int64

### 3.1. Article 42 (No Matches)

In [11]:
full_data['art42_charter'].value_counts()

art42_charter
0    55065
Name: count, dtype: int64

### 3.2. Article 25

In [12]:
# number resolutions with reference to Article 25
full_data[full_data.art25_charter == 1].doc.nunique()

30

In [13]:
# number of paragraphs with reference to Article 25
full_data[full_data.art25_charter == 1].shape[0]

36

### 3.3. Chapter VII

In [14]:
# number of Resolutions with ref to Chapter VII 
full_data[(full_data.chpVII == 1)].doc.nunique()

896

In [15]:
# resolutions with reference to Chapter VII - same result with different df ;) 
dec_data[dec_data.chpVII == 1].shape[0]

896

In [16]:
# number of clauses with ref to Chapter VII 
full_data[(full_data.chpVII == 1)].shape[0]

921

In [17]:
full_data[(full_data.chpVII == 1)]['segmentClass'].value_counts()

segmentClass
content        812
content_num    109
Name: count, dtype: int64

### 3.4. Article 41

In [18]:
# clauses with Art. 41
full_data.art41_charter.value_counts()

art41_charter
0    55002
1       63
Name: count, dtype: int64

In [19]:
# resolutions with Art. 41
full_data[full_data.art41_charter == 1].doc.nunique()

49

### 3.5. Article 39

In [20]:
# resolutions Art. 39
full_data[(full_data.art39_charter == 1)].doc.to_list()

['S_RES_0054_1948',
 'S_RES_0054_1948',
 'S_RES_0062_1948',
 'S_RES_0232_1966',
 'S_RES_0461_1979',
 'S_RES_0598_1987',
 'S_RES_0660_1990']

### 3.6. Article 39 language

In [21]:
# resolutions with ref to Art. 39
full_data[(full_data.art39lang == 1)].doc.nunique()

1100

In [22]:
full_data[(full_data.art39lang == 1)]['segmentClass'].value_counts()

segmentClass
content        1451
content_num     265
Name: count, dtype: int64

### 3.7. "Breach of International Peace"

In [23]:
print(full_data.breach_int.sum())

1


In [24]:
full_data[full_data.breach_int == 1].iloc[0,1]

'Determining that there exists a breach of international peace and security as regards the Iraqi invasion of Kuwait'

### 3.8. Decides / Demand / Authorizes

In [25]:
# resolutions with "Decides to remain actively …"
full_data[(full_data.decides_act == 1)].doc.nunique()

1743

In [26]:
# resoltions with "decides" (without "decides_act")
full_data[(full_data.decides == 1)].doc.nunique()

1820

In [27]:
# resoltions with "demand" (without "decides_act")
full_data[(full_data.demand == 1)].doc.nunique()

572

In [28]:
# resoltions with "authorizes"
full_data[(full_data.authorizes == 1)].doc.nunique()

295

### 3.9. Subset: Demand Request Decide

In [29]:
tmp_data = dec_data[(dec_data.art39lang == 1) &
                    (dec_data.chpVII == 0) &
                    (dec_data.art25_charter == 0) &
                    (dec_data.art39_charter == 0) &
                    (dec_data.art40_charter == 0) & 
                    (dec_data.art41_charter == 0)]

# resolutions with art39 language that do not contain an explicit reference to Chapter VII, Art. 25, Art. 39, Art. 40, Art. 41.
tmp_data.doc.nunique()

377

In [30]:
tmp_data = full_data[full_data.doc.isin(tmp_data.doc.to_list())]
tmp_data = tmp_data[(tmp_data["demand"] == 1) | (tmp_data["requests"] == 1) | (tmp_data["decides"] == 1)]
print(tmp_data.shape)

# resolutions with art39 language and either "demand", "requests" or "decides" but do not contain an explicit reference to Chapter VII, Art. 25, Art. 39, Art. 40, Art. 41.
tmp_data.doc.nunique()

(1235, 21)


347

In [31]:
tmp_data.to_pickle("data/subset_demand_request_decide.pkl")

### 3.10. Demand in resolutions without Decides / Chapter 7

In [32]:
dec_data[(dec_data.demand == 1) & ((dec_data.decides == 0) & (dec_data.chpVII == 0))].doc.nunique()

123

In [33]:
# resolutions with a reference to Chapter VII also contain a decision other than to remain seized of the matter
dec_data[dec_data.chpVII == 1].decides.value_counts()

decides
1    823
0     73
Name: count, dtype: int64

In [34]:
# resolution that contain a demand: 301 also contain a refernce to Chapter VII; 271 do not
dec_data[dec_data.demand == 1].chpVII.value_counts()

chpVII
1    301
0    271
Name: count, dtype: int64